# WISER Module 7 — Deep Learning and Images

This notebook follows **Rise Sections 7.3–7.4**. The image exercise uses **Fashion-MNIST clothing images** to learn CNN methods. It does **not** test facial, pupil, or adolescent return-to-use predictions. Complete one pathway:

- **Foundational (7.31):** Build a small TensorFlow/Keras CNN, interpret training/validation curves and test errors, complete a ~250-word reflection, and submit the protocol sketch.
- **Applied (7.32):** Adapt the tutorial to a documented openly licensed dataset, implement and justify an extension, complete a 1–2 page model card, a ~500-word appraisal of the study attached in Rise, and the protocol sketch.

**Start:** Select **File → Save a copy in Drive**. Run cells in order, then replace all bracketed responses. Running everything without editing does **not** finish the assignment. Save your completed `.ipynb`, the required PDF(s), and the outputs ZIP. Put your name or learner ID where prompted.

**Privacy:** Never upload identifiable facial/ocular images, PHI, or sensitive participant images into this notebook. A successful clothing classifier provides no evidence for clinical use.

In [ ]:
PATHWAY = "Foundational"  # Change to "Applied" only if you chose 7.32 in Rise.
LEARNER_NAME = "[Your name or learner ID]"
assert PATHWAY in {"Foundational", "Applied"}
print("Pathway:", PATHWAY, "| Learner:", LEARNER_NAME)

## 1. Setup and dataset provenance (Rise 7.4: explore)

A **CNN must actually train** for this activity. If TensorFlow is unavailable, the cell gives a clear installation instruction and stops; it does not substitute random arrays or imply a model was evaluated. In Colab, run `%pip install tensorflow` in a new code cell if necessary, restart the runtime, and rerun the notebook. A CPU runtime works with the small subset below; a GPU may be faster.

[TensorFlow's Fashion-MNIST introduction](https://www.tensorflow.org/tutorials/keras/classification) describes the clothing dataset and its ten classes. [TensorFlow's CNN tutorial](https://www.tensorflow.org/tutorials/images/cnn) provides the model pattern. The images are methods practice, **not a clinical proxy**.

In [ ]:
from pathlib import Path
import json, re, shutil, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
try:
    from IPython.display import display
except ImportError:
    display = print
try:
    import tensorflow as tf
    from tensorflow import keras
except ImportError as exc:
    raise RuntimeError("TensorFlow is required for Module 7. In Colab run %pip install tensorflow in a separate code cell, restart Runtime, and rerun from the top.") from exc

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
OUT = Path("module7_outputs")
OUT.mkdir(exist_ok=True)
print("TensorFlow:",tf.__version__)

### Applied dataset choice

The default Fashion-MNIST run supports **Foundational** and **Applied offline development only**. To complete Applied, adapt the input/model to an openly licensed dataset you choose, document its source and license, and rerun evaluation. An approved external dataset needs separate training, validation, and held-out test splits. Do not upload clinical images whose use or redistribution is not permitted. The form below makes the unresolved Applied items visible.

In [ ]:
DATASET_NAME = "Fashion-MNIST"
DATASET_REFERENCE = "https://www.tensorflow.org/tutorials/keras/classification"
DATASET_LICENSE_NOTE = "[Verify permitted use and document license/attribution before redistributing images or adapting a different dataset.]"
APPLIED_DATASET_SOURCE = ""  # Applied final: name/reference to your documented openly licensed replacement.
APPLIED_ADAPTATION_NOTES = """[Applied: document which loader, preprocessing and model cells you changed for your chosen dataset.]"""
if PATHWAY == "Applied" and not APPLIED_DATASET_SOURCE:
    print("APPLIED OFFLINE PRACTICE ONLY: replace Fashion-MNIST with your documented dataset before final submission.")

In [ ]:
(images_all, labels_all), (images_official_test, labels_official_test) = keras.datasets.fashion_mnist.load_data()
CLASS_NAMES = ["T-shirt/top","Trouser","Pullover","Dress","Coat","Sandal","Shirt","Sneaker","Bag","Ankle boot"]
train_rows, val_rows = train_test_split(np.arange(len(images_all)),train_size=12000,test_size=2000,
                                        random_state=SEED,stratify=labels_all)
test_rows, _ = train_test_split(np.arange(len(images_official_test)),train_size=2000,
                                random_state=SEED,stratify=labels_official_test)
x_train = images_all[train_rows].astype("float32")[...,np.newaxis]/255.0
y_train = labels_all[train_rows]
x_val = images_all[val_rows].astype("float32")[...,np.newaxis]/255.0
y_val = labels_all[val_rows]
x_test = images_official_test[test_rows].astype("float32")[...,np.newaxis]/255.0
y_test = labels_official_test[test_rows]
print("Separate train/validation/test image counts:",len(x_train),len(x_val),len(x_test))
print("Image tensor (height, width, channels):",x_train.shape[1:])
class_counts=pd.DataFrame({"class":CLASS_NAMES,
    "train_n":np.bincount(y_train,minlength=10),
    "validation_n":np.bincount(y_val,minlength=10),
    "test_n":np.bincount(y_test,minlength=10)})
display(class_counts)

## 2. Inspect sample images and labels

Each image is 28×28 grayscale pixels; normalization scales values from 0–255 to 0–1. Labels describe **clothing categories**. Look for similar-looking classes and consider what an accuracy score would hide. The image gallery is from training data only and is displayed in the notebook without exporting source images.

In [ ]:
fig,axes=plt.subplots(2,5,figsize=(9,4))
for ax,img,label_id in zip(axes.ravel(),x_train[:10],y_train[:10]):
    ax.imshow(img.squeeze(),cmap="gray",vmin=0,vmax=1)
    ax.set_title(CLASS_NAMES[int(label_id)],fontsize=8)
    ax.axis("off")
fig.suptitle("Fashion-MNIST practice images (clothing, not clinical images)")
fig.tight_layout()
plt.show()
class_observation = """[Identify one pair of clothing classes that might be confused and why.]"""
alt_text = """[Write alt text describing the gallery of clothing examples and their class labels.]"""
print(class_observation)

## 3. Build and train a small CNN (Rise 7.31)

Convolutional layers learn local patterns; pooling reduces spatial size; the final layer predicts one of ten clothing labels. Training images update weights. Validation images monitor learning without updating weights. The **official test images stay held out** until the next section. For Applied, the example augmentation demonstrates one permitted extension; decide whether flips and rotations preserve label meaning for *your* dataset.

In [ ]:
layers=[keras.layers.Input(shape=(28,28,1))]
if PATHWAY == "Applied":
    layers += [keras.layers.RandomFlip("horizontal"),keras.layers.RandomRotation(0.04)]
    print("Applied example extension: light augmentation; justify it for your chosen images.")
layers += [
    keras.layers.Conv2D(16,3,activation="relu"),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(32,3,activation="relu"),
    keras.layers.MaxPooling2D(),
    keras.layers.Flatten(),
    keras.layers.Dense(32,activation="relu"),
    keras.layers.Dense(len(CLASS_NAMES),activation="softmax"),
]
model=keras.Sequential(layers)
model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])
model.summary()
EPOCHS=3
history=model.fit(x_train,y_train,validation_data=(x_val,y_val),
                  batch_size=64,epochs=EPOCHS,verbose=2)

## 4. Inspect training and validation curves

**Your response:** Describe what happened to training and validation accuracy and loss. Overfitting would appear if training keeps improving while validation stops improving or gets worse. Three epochs may not display overfitting; say that explicitly if the curves do not show it. Do not infer external or clinical validity from either curve.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10,3.7))
for ax,metric,title in zip(axes,["accuracy","loss"],["Accuracy","Loss"]):
    ax.plot(history.history[metric],marker="o",label="Training")
    ax.plot(history.history["val_"+metric],marker="o",label="Validation")
    ax.set(xlabel="Epoch",title=title)
    ax.set_xticks(range(EPOCHS),[str(i+1) for i in range(EPOCHS)])
    ax.grid(axis="y",alpha=.3)
    ax.legend(frameon=False)
fig.tight_layout()
curves_path=OUT/"training_validation_curves.png"
fig.savefig(curves_path,dpi=170,bbox_inches="tight")
plt.show()
curve_response = """[Describe both curves. Did they show overfitting? What could more epochs or another dataset reveal?]"""
print(curve_response)

## 5. Held-out test and errors (Rise 7.4: evaluate)

Test accuracy is a within-dataset result. Inspect the confusion matrix and class-specific precision/recall; name a concrete confusion or limitation. It does **not** validate use with adolescents, different devices, or clinical images. External and prospective validation would be separate studies with permission and safeguards.

In [ ]:
test_loss,test_acc=model.evaluate(x_test,y_test,verbose=0)
prob=model.predict(x_test,batch_size=128,verbose=0)
pred=np.argmax(prob,axis=1)
cm=confusion_matrix(y_test,pred,labels=np.arange(len(CLASS_NAMES)))
report=classification_report(y_test,pred,target_names=CLASS_NAMES,output_dict=True,zero_division=0)
print(f"Fashion-MNIST held-out test accuracy: {test_acc:.3f}; loss: {test_loss:.3f}")
display(pd.DataFrame(report).T.round(3))
fig,ax=plt.subplots(figsize=(7,6))
im=ax.imshow(cm,cmap="Blues")
ax.set(xticks=np.arange(10),yticks=np.arange(10),xticklabels=CLASS_NAMES,yticklabels=CLASS_NAMES,
       xlabel="Predicted clothing class",ylabel="Actual clothing class",title="Held-out clothing test errors")
plt.setp(ax.get_xticklabels(),rotation=50,ha="right")
fig.colorbar(im,ax=ax,label="Images")
fig.tight_layout()
confusion_path=OUT/"confusion_matrix.png"
fig.savefig(confusion_path,dpi=170,bbox_inches="tight")
plt.show()
error_response = """[Name one frequently confused class pair from the matrix or one low-recall class. Explain why accuracy alone would miss it.]"""
print(error_response)

## 6. Link the technical steps to the HopeBridge scenario

The HopeBridge vendor scenario involves **adolescent facial/ocular images**, a proposed return-to-use label, consent and privacy, and continuity of care. Fashion-MNIST cannot test any of those clinical claims. Write one sentence each on (1) what the held-out clothing test checks, (2) what a different site/device/adolescent group would require, (3) why image access and consent matter, and (4) how care would continue regardless of any research score.

In [ ]:
scenario_connection = """[Write four short sentences connecting training/test evaluation, external validation, privacy/consent, and continuity of care to the HopeBridge scenario.]"""
print(scenario_connection)

## 7. Inquiry Design Challenge protocol sketch (Rise 7.3)

**Complete the protocol sketch even though the CNN uses clothing images.** Draft an ethically defensible *proposed study* at HopeBridge. In concise responses, address the research question, dataset/population/licensing, task and model, internal/external/prospective validation, privacy across collection/storage/access/retention/deletion, stop conditions, and unchanged access to care. The sketch does **not** authorize collecting images or using scores in clinical care.

The notebook exports your responses as a PDF and a Markdown file. If your instructor requires the downloadable **Rise DOCX protocol-sketch template**, transfer these answers into that template as well.

In [ ]:
protocol = {
 "research_question":"[One stigma-free research question for a proposed study; specify the proposed task, not a claimed clinical predictor.]",
 "dataset_card":"[Describe proposed population, dataset provenance, licensing/permissions, FAIR metadata and access limits. Fashion-MNIST is methods practice only.]",
 "model_and_task":"[Classification, detection, or regression? Explain the outcome and why the task fits it.]",
 "validation":"[Internal holdout, external adolescent/device/site validation, and prospective research-only validation if appropriate.]",
 "privacy_pipeline":"[Collection/permission and assent review as applicable; approved storage; role-based access; retention and deletion; no public identifiable images.]",
 "stop_conditions":"[State evidence or a privacy/consent concern that would halt the research or prohibit clinical use.]",
 "continuity_of_care":"[Explain how care and services continue regardless of research participation or score.]",
}
display(pd.Series(protocol))

## 8. Pathway-specific work

**Foundational:** Write ~250 words explaining the training/validation/test steps and one model error, then map those observations to the HopeBridge decision about external validation, privacy, and unchanged care.

**Applied:** Use an openly licensed dataset you chose, document its license and adaptation, justify augmentation/transfer learning/imbalance handling, write a 1–2 page model card, and critically appraise the **smartphone eye-scanning paper attached in Rise** (~500 words). Read that paper before writing; distinguish what it actually studied from the HopeBridge proposal. The placeholders below are prompts, not a substitute for adapting the dataset.

In [ ]:
foundational_reflection = """[Foundational: ~250 words linking the CNN steps and errors to the HopeBridge validation/privacy/continuity decisions.]"""
applied_appraisal = """[Applied: ~500-word critical appraisal of the paper in Rise. Address population, outcome, validation, device/setting transfer, privacy, and clinical-use limits with source-specific details.]"""
if PATHWAY == "Foundational": print("Reflection words:",len(foundational_reflection.split()))
else: print("Appraisal words:",len(applied_appraisal.split()))

### Applied model card (Rise 7.32)

Applied learners complete every field and export a 1–2 page PDF. **Out-of-scope use must explicitly include adolescent clinical deployment unless separately validated and approved.** These entries should describe the dataset actually used, not a pretend clinical model.

In [ ]:
model_card = {
 "intended_use":"[Applied: describe clothing/open-image methods practice and who may use these research outputs.]",
 "out_of_scope":"[Explicitly exclude adolescent clinical deployment and any claim that a clothing classifier predicts return to use.]",
 "training_data":"[Chosen dataset, provenance/license, population or image source, classes, preprocessing, and splits.]",
 "evaluation":"[Report your actual validation/test accuracy, class distribution, confusion matrix and failure mode.]",
 "limitations":"[Dataset shift, label limits, generalizability, privacy, bias, and missing external/prospective validation.]",
 "governance":"[Human oversight and evidence/permission needed before any other application.]",
}
if PATHWAY == "Applied":display(pd.Series(model_card))

## 9. Export and check your submission

Run this cell **after** completing responses. It creates a one-page protocol PDF for both pathways, plus an Applied model-card PDF (1–2 pages), Markdown responses, charts, metadata, and a ZIP. The required notebook still needs to be saved separately as `.ipynb`. The paper appraisal is written by the Applied learner after reading the attached Rise paper. Open each PDF to check readability and the learner identifier before submitting.

In [ ]:
def write_sections_pdf(path,title,sections,max_chars=240):
    fig=plt.figure(figsize=(8.5,11),facecolor="white")
    fig.text(.075,.965,title,fontsize=15,weight="bold",color="#174A59")
    fig.text(.075,.937,f"Learner: {LEARNER_NAME}  |  WISER Module 7",fontsize=9)
    y=.903
    for heading,value in sections.items():
        if len(value)>max_chars: raise ValueError(f"Shorten '{heading}' for the PDF (max {max_chars} characters).")
        lines=textwrap.wrap(value,width=98) or [""]
        fig.text(.075,y,heading.replace("_"," ").title(),fontsize=10.4,weight="bold",va="top")
        fig.text(.075,y-.022,"\n".join(lines),fontsize=9,va="top",linespacing=1.22)
        y-=.022+.019*len(lines)+.026
    if y < .075: raise ValueError("Shorten your responses to keep the PDF readable on one page.")
    fig.text(.075,.032,"Research planning exercise; no clinical deployment or identifiable images",fontsize=7.5)
    with PdfPages(path) as pdf:pdf.savefig(fig)
    plt.close(fig)
    return path

protocol_path=write_sections_pdf(OUT/"Module_7_protocol_sketch.pdf","Module 7 | Protocol sketch",protocol,max_chars=230)
print("Saved protocol PDF:",protocol_path)
model_card_path=None
if PATHWAY == "Applied":
    model_card_path=write_sections_pdf(OUT/"Module_7_model_card.pdf","Module 7 | Model card",model_card,max_chars=310)
    print("Saved Applied model-card PDF:",model_card_path)

In [ ]:
responses={"class_observation":class_observation,"gallery_alt_text":alt_text,
           "training_validation":curve_response,"held_out_errors":error_response,
           "scenario_connection":scenario_connection,
           "pathway_reflection_or_appraisal":foundational_reflection if PATHWAY=="Foundational" else applied_appraisal}
if PATHWAY=="Applied":responses.update({"dataset_license":DATASET_LICENSE_NOTE,
  "dataset_adaptation":APPLIED_ADAPTATION_NOTES,"dataset_source":APPLIED_DATASET_SOURCE,
  **{f"model_card_{k}":v for k,v in model_card.items()}})
unfinished=[k for k,v in {**responses,**protocol,"learner":LEARNER_NAME}.items() if "[" in str(v) and "]" in str(v)]
if unfinished:print("TO COMPLETE before submitting:",", ".join(unfinished))
words=len(foundational_reflection.split()) if PATHWAY=="Foundational" else len(applied_appraisal.split())
low,high=(220,280) if PATHWAY=="Foundational" else (450,550)
if not low<=words<=high:print(f"TO COMPLETE: write about {250 if PATHWAY=='Foundational' else 500} words for your pathway.")
if PATHWAY=="Applied" and not APPLIED_DATASET_SOURCE:
    print("APPLIED: a documented openly licensed replacement dataset and actual adaptation are required for final submission.")

(OUT/"protocol_sketch.md").write_text("# Module 7 protocol sketch\n\n"+
    "\n\n".join(f"## {k.replace('_',' ').title()}\n{v}" for k,v in protocol.items())+"\n",encoding="utf-8")
(OUT/"responses.md").write_text("# Learner responses\n\n"+
    "\n\n".join(f"## {k.replace('_',' ').title()}\n{v}" for k,v in responses.items())+"\n",encoding="utf-8")
if PATHWAY=="Applied":
    (OUT/"model_card.md").write_text("# Model card\n\n"+
        "\n\n".join(f"## {k.replace('_',' ').title()}\n{v}" for k,v in model_card.items())+"\n",encoding="utf-8")
    (OUT/"appraisal_memo.md").write_text("# Paper appraisal\n\n"+applied_appraisal+"\n",encoding="utf-8")
else:
    (OUT/"foundational_reflection.md").write_text("# Foundational reflection\n\n"+foundational_reflection+"\n",encoding="utf-8")
metadata={"module":"WISER 7","pathway":PATHWAY,"learner":LEARNER_NAME,
 "training_dataset":DATASET_NAME,"dataset_reference":DATASET_REFERENCE,
 "image_task":"clothing image classification","cnn_trained":True,"train_n":len(x_train),"val_n":len(x_val),"test_n":len(x_test),
 "epochs":EPOCHS,"test_accuracy":float(test_acc),"test_loss":float(test_loss),"seed":SEED,
 "clinical_relevance":"None; not an adolescent or pupil/facial imaging model", "source_images_exported":False}
(OUT/"metadata.json").write_text(json.dumps(metadata,indent=2),encoding="utf-8")
(OUT/"README.md").write_text(
 f"# WISER Module 7 outputs\n\nPathway: {PATHWAY}. Learner: {LEARNER_NAME}.\n\n"
 "This is Fashion-MNIST clothing classification for CNN methods practice. The held-out test is from the same image source; "
 "there is no external/prospective clinical validation. Outputs include aggregate curves and error summaries, "
 "protocol sketch, learner responses, and pathway-specific documents. No patient images or individual predictions are exported. "
 "An Applied final submission requires its own documented openly licensed dataset and adaptation.\n",encoding="utf-8")
archive=shutil.make_archive("WISER_Module_7_outputs","zip",root_dir=".",base_dir=str(OUT))
print("ZIP:",archive,"| files:",sorted(p.name for p in OUT.iterdir()))
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Outside Colab: download the ZIP from the file browser.")